# PCam binary classifier — Kaggle T4 GPU
Training ResNet-18 on PatchCamelyon for tumour/normal patch classification.
Artifacts are saved to `/kaggle/working/` and can be downloaded after the run.

In [15]:
import os, time, json, logging
from dataclasses import dataclass
from pathlib import Path
from typing import Tuple

import h5py
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from sklearn.metrics import roc_auc_score, f1_score, confusion_matrix
from PIL import Image

logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger(__name__)

print(f'PyTorch: {torch.__version__}')
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

DATA_DIR = Path('/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon')
OUTPUT_DIR = Path('/kaggle/working/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Confirm HDF5 files are present
for f in sorted(DATA_DIR.rglob('*.h5')):
    print(f)

PyTorch: 2.10.0+cu128
CUDA: True
GPU: Tesla T4
/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_test_y.h5
/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_train_y.h5
/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/Labels/Labels/camelyonpatch_level_2_split_valid_y.h5
/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/camelyonpatch_level_2_split_train_mask/camelyonpatch_level_2_split_train_mask.h5
/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/pcam/test_split.h5
/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/pcam/training_split.h5
/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon/pcam/validation_split.h5


In [17]:
import h5py
BASE = Path('/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon')

DATA_TRAIN_X = BASE / 'pcam/training_split.h5'
DATA_TRAIN_Y = BASE / 'Labels/Labels/camelyonpatch_level_2_split_train_y.h5'
DATA_VAL_X   = BASE / 'pcam/validation_split.h5'
DATA_VAL_Y   = BASE / 'Labels/Labels/camelyonpatch_level_2_split_valid_y.h5'
with h5py.File(DATA_TRAIN_X, 'r') as f:
    print('Train X keys:', list(f.keys()))
    print('Shape:', f[list(f.keys())[0]].shape)

with h5py.File(DATA_TRAIN_Y, 'r') as f:
    print('Train Y keys:', list(f.keys()))
    print('Shape:', f[list(f.keys())[0]].shape)

Train X keys: ['x']
Shape: (262144, 96, 96, 3)
Train Y keys: ['y']
Shape: (262144, 1, 1, 1)


In [ ]:
@dataclass(frozen=True)
class TrainingConfig:
    output_dir: str        = '/kaggle/working/checkpoints'
    epochs: int            = 5
    batch_size: int        = 128
    learning_rate: float   = 1e-4
    num_workers: int       = 2
    log_every_n_steps: int = 200

    @property
    def device(self) -> torch.device:
        return torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BASE         = Path('/kaggle/input/datasets/andrewmvd/metastatic-tissue-classification-patchcamelyon')
TRAIN_X      = BASE / 'pcam/training_split.h5'
TRAIN_Y      = BASE / 'Labels/Labels/camelyonpatch_level_2_split_train_y.h5'
VAL_X        = BASE / 'pcam/validation_split.h5'
VAL_Y        = BASE / 'Labels/Labels/camelyonpatch_level_2_split_valid_y.h5'
OUTPUT_DIR   = Path('/kaggle/working/checkpoints')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


class PCamHDF5Dataset(Dataset):
    def __init__(self, x_path, y_path, transform):
        self.x_path    = x_path
        self.y_path    = y_path
        self.transform = transform
        with h5py.File(x_path, 'r') as f:
            self.length = f['x'].shape[0]

    def __len__(self):
        return self.length

    def __getitem__(self, idx):
        with h5py.File(self.x_path, 'r') as fx, h5py.File(self.y_path, 'r') as fy:
            image = Image.fromarray(fx['x'][idx])
            label = int(fy['y'][idx][0][0][0])
        return self.transform(image), label


def get_transforms():
    mean = [0.485, 0.456, 0.406]
    std  = [0.229, 0.224, 0.225]
    train_tf = transforms.Compose([
        transforms.RandomHorizontalFlip(),
        transforms.RandomVerticalFlip(),
        transforms.ColorJitter(brightness=0.2, contrast=0.2),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    val_tf = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    return train_tf, val_tf


def get_loaders(cfg):
    train_tf, val_tf = get_transforms()
    train_ds = PCamHDF5Dataset(TRAIN_X, TRAIN_Y, train_tf)
    val_ds   = PCamHDF5Dataset(VAL_X,   VAL_Y,   val_tf)
    log.info(f'Train: {len(train_ds)} samples, Val: {len(val_ds)} samples')
    train_loader = DataLoader(
        train_ds, batch_size=cfg.batch_size,
        shuffle=True, num_workers=cfg.num_workers, pin_memory=True,
    )
    val_loader = DataLoader(
        val_ds, batch_size=cfg.batch_size,
        shuffle=False, num_workers=cfg.num_workers, pin_memory=True,
    )
    return train_loader, val_loader


def build_model():
    model    = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
    model.fc = nn.Linear(model.fc.in_features, 2)
    return model


def train_epoch(model, loader, criterion, optimizer, device, log_every):
    model.train()
    total_loss = 0.0
    for step, (images, labels) in enumerate(loader):
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        if step % log_every == 0:
            log.info(f'  step {step}/{len(loader)}  loss={loss.item():.4f}')
    return total_loss / len(loader)


def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, all_labels, all_probs, all_preds, latencies = 0.0, [], [], [], []
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            t0     = time.perf_counter()
            logits = model(images)
            latencies.append((time.perf_counter() - t0) * 1000)
            total_loss += criterion(logits, labels).item()
            all_probs.extend(torch.softmax(logits, 1)[:, 1].cpu().numpy())
            all_preds.extend(logits.argmax(1).cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
    all_labels = np.array(all_labels)
    all_preds  = np.array(all_preds)
    all_probs  = np.array(all_probs)
    return {
        'loss':                 round(total_loss / len(loader), 4),
        'accuracy':             round(float((all_preds == all_labels).mean()), 4),
        'auc':                  round(float(roc_auc_score(all_labels, all_probs)), 4),
        'f1':                   round(float(f1_score(all_labels, all_preds)), 4),
        'confusion_matrix':     confusion_matrix(all_labels, all_preds).tolist(),
        'latency_ms_per_batch': round(float(np.mean(latencies)), 2),
    }


# --- Main training loop ---
cfg       = TrainingConfig()
device    = cfg.device
model     = build_model().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=cfg.learning_rate)
train_loader, val_loader = get_loaders(cfg)

best_auc, all_metrics = 0.0, []

for epoch in range(1, cfg.epochs + 1):
    log.info(f'Epoch {epoch}/{cfg.epochs}')
    train_loss  = train_epoch(model, train_loader, criterion, optimizer, device, cfg.log_every_n_steps)
    val_metrics = evaluate(model, val_loader, criterion, device)
    val_metrics.update({'epoch': epoch, 'train_loss': round(train_loss, 4)})
    all_metrics.append(val_metrics)
    log.info(
        f"  val_loss={val_metrics['loss']}  "
        f"acc={val_metrics['accuracy']}  "
        f"auc={val_metrics['auc']}  "
        f"f1={val_metrics['f1']}"
    )
    if val_metrics['auc'] > best_auc:
        best_auc = val_metrics['auc']
        torch.save(model.state_dict(), OUTPUT_DIR / 'best_model.pt')
        log.info(f'  New best AUC={best_auc:.4f} — checkpoint saved')

torch.save(model.state_dict(), OUTPUT_DIR / 'final_model.pt')

with open(OUTPUT_DIR / 'metrics.json', 'w') as f:
    json.dump(all_metrics, f, indent=2)
with open(OUTPUT_DIR / 'config.json', 'w') as f:
    json.dump(cfg.__dict__, f, indent=2)

print('Training complete!')
print(json.dumps(all_metrics[-1], indent=2))

2026-04-12 09:50:46,108 INFO Train: 262144 samples, Val: 32768 samples
2026-04-12 09:50:46,110 INFO Epoch 1/5
2026-04-12 09:50:54,359 INFO   step 0/2048  loss=0.7737
2026-04-12 09:58:16,307 INFO   step 200/2048  loss=0.1809
2026-04-12 10:06:08,399 INFO   step 400/2048  loss=0.1444
2026-04-12 10:13:25,684 INFO   step 600/2048  loss=0.1377


In [ ]:
# List output artifacts
for f in sorted(OUTPUT_DIR.iterdir()):
    size_mb = f.stat().st_size / 1024 / 1024
    print(f'{f.name:30s}  {size_mb:.1f} MB')